# **CMP-7023B Data Mining - Advanced Data Analysis**
## **100318577**

### **Aim:**
To analyse the `Insurance_Data.csv` dataset to understand better customer profiling and behaviour prediction in the insurance industry. This process focuses on classifying customers into predefined subtypes using supervised learning techniques and segmenting them into natural clusters through unsupervised learning methods. Comprising of 5,521 observations and 83 variables, including household size, education level, income, social class, and contributions to various insurance policies, such as car, life, disability, and property insurance. This task comprehensively evaluates these attributes to identify patterns and relationships that can inform strategic decision-making. The goal is to perform detailed data exploration and preprocessing and apply machine learning algorithms to extract actionable insights. These findings will be interpreted and presented in a structured report tailored for insurance executives, highlighting opportunities for personalized services and improved policy offerings.

**Primary goals:**

1. **Classification**: Using supervised learning techniques to predict and categorise customers into relevant subtypes, such as "Rural & Low-Income" or "Wealthy & Affluent."
2. **Segmentation**: Applying unsupervised learning methods to explore natural groupings among customer profiles, independent of predefined categories.

**Final Workflow Summary**

1. **Data Dictionary**
 - Loading insurance data
 - Variable standardisation and renaming to be more descriptive and representative of their contents.
 - Generate descriptive statistics
   
2. **Categorical and Continuous variables**
 - Group categorical and continuous variables for later analysis
   
3. **Handling missing Data**
- Categorical variables were replaced with the mode to preserve the discrete patterns inherent in these data.
- Numerical variables were imputed using the median to mitigate the impact of outliers while maintaining the dataset’s statistical balance.  

4. **Outlier Detection (DBSCAN)**
   - Identifies and removes noise for cleaner data.

5. **Unsupervised Clustering (DBSCAN and Hierarchical Clustering)**
   - Segments data into meaningful groups.
   
6. **Supervised Learning (Decision Trees, Random Forests and KNN)**
   - Uses clustered data for high-accuracy predictions.

# 1. Data Dictionary for Insurance Data

The following code generates a **comprehensive data dictionary** for the `insurance` dataset, organised with key information into a markdown table with standardised and refined variable names, ensuring they are more meaningful and representative of the data's content. The process involves:

1.1. **Variable Standardisation**: Original variable names from the dataset are updated and refined to be more descriptive and representative of their contents. This standardisation improves clarity and usability. **Missing** data is identified, calculated and formatted as a number and percentage of missing data per variable.

1.2. **Descriptions**: Generate descriptions based on column names and contents.

1.3. **Descriptive Statistics**: Key statistical metrics are calculated for each column, including mean, minimum, maximum, standard deviation, and percentiles (25%, 50%, 75%). **Categorical Variables** (Non-numeric) are processed to extract meaningful insights. The "Mean" is represented by the mode (most frequent value), while "Min" and "Max" are determined by the smallest and largest non-null values. Missing values are replaced with `Na`. **Numerical Variables** metrics such as mean, min, max, and percentiles are calculated, rounded to two decimal places, and formatted to display whole numbers as integers.

1.4. **Example Values**: Representative examples (up to five unique values) are drawn from each column and formatted for clarity. Whole numeric values (e.g., `1.0`, `2.0`) are displayed as integers (`1`, `2`).

1.4. **Data Dictionary Compilation**: All gathered information—including refined variable names, data types, descriptions, example values, and statistical summaries—is compiled into a structured dictionary, which is converted into a markdown table for presentation.

### Markdown Table Output:
The final markdown table includes the following headings:
- **Variable Name**: Standardised and refined column names for clarity.
- **Original Name**: The original column names from the dataset.
- **Type**: Data types such as numeric or categorical (e.g., `int64`, `object`).
- **Number of Entries**: Count of non-null values for each variable.
- **Description**: A meaningful explanation of the column’s purpose and content.
- **Example Values**: Up to five unique representative values, formatted for readability.
- **Mean, Std Dev, Min, Max, Percentiles (25%, 50%, 75%)**: Statistical summaries for numeric columns, with categorical columns displaying mode, minimum, and maximum categories instead of NaN.

[Disclaimer]: The following code for Data Dictionary generation was supported by Microsoft Copilot/Claude.ai, an AI-powered assistant to assist in Python code generation and debugging.

In [1]:
# PACKAGES ----
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from scipy import stats

# Suppress warnings
import warnings # reduce cluttered output
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# LOAD DATA
insurance = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data_mining_2/Insurance_Data.csv")

print(insurance.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5521 entries, 0 to 5520
Data columns (total 83 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Customer_Type                                    5521 non-null   object 
 1   Number_of_Houses                                 5521 non-null   int64  
 2   Avg_Household_Size                               5521 non-null   int64  
 3   Avg_Age                                          5507 non-null   object 
 4   Household_Profile                                5521 non-null   object 
 5   Married                                          5521 non-null   int64  
 6   Living_Together                                  5521 non-null   int64  
 7   Other_Relation                                   5521 non-null   int64  
 8   Singles                                          5521 non-null   int64  
 9   Household_Without_Children    

In [3]:
# Create a dictionary to store original variable names
original_names = {}

# Define mapping between original column names and new cleaner names
column_mapping = {
    # Household and customer demographics
    'customer_type': 'customer_type',
    'number_of_houses': 'total_properties_owned',
    'avg_household_size': 'avg_household_size',
    'avg_age': 'avg_age',
    'household_profile': 'household_classification',
    'married': 'married',
    'living_together': 'living_together',
    'other_relation': 'extended_family_members',
    'singles': 'singles',
    'household_without_children': 'childless_households',
    'household_with_children': 'households_with_children',
    # Education Levels
    'high_education_level': 'highly_educated',
    'medium_education_level': 'moderately_educated',
    'low_education_level': 'low_education',
    # Occupation Categories
    'high_status': 'high_status_professionals',
    'entrepreneur': 'entrepreneurs',
    'farmer': 'farmer',
    'middle_management': 'middle_management',
    'skilled_labourers': 'skilled_workers',
    'unskilled_labourers': 'unskilled_workers',
    # Social Class Categories
    'social_class_a': 'social_class_a',
    'social_class_b1': 'social_class_b1',
    'social_class_b2': 'social_class_b2',
    'social_class_c': 'social_class_c',
    'social_class_d': 'social_class_d',
    # Housing and vehicle ownership
    'rented_house': 'rented_house',
    'home_owner': 'home_owner',
    'owns_one_car': 'single_car',
    'owns_two_cars': 'dual_car',
    'owns_no_car': 'no_car',
    # Health insurance
    'national_health_insurance': 'national_health_insurance',
    'private_health_insurance': 'private_health_insurance',
    # Income categories
    'income_less_than_30k': 'income_below_30k',
    'income_30k_to_45k': 'income_30k_to_45k',
    'income_45k_to_75k': 'income_45k_to_75k',
    'income_75k_to_122k': 'income_75k_to_122k',
    'income_above_123k': 'income_above_123k',
    'average_income': 'mean_income',
    'purchasing_power_class': 'purchasing_power_group',
    # Financial
    'private_third_party_insurance_contribution': 'private_third_party',
    'business_third_party_insurance_contribution': 'business_third_party',
    'agricultural_third_party_insurance_contribution': 'agricultural_third_party',
    'car_policy_contribution': 'car_insurance',
    'delivery_van_policy_contribution': 'van_insurance',
    'motorcycle_scooter_policy_contribution': 'bike_insurance',
    'lorry_policy_contribution': 'lorry_insurance',
    'trailer_policy_contribution': 'trailer_insurance',
    'tractor_policy_contribution': 'tractor_insurance',
    'agricultural_machine_policy_contribution': 'agrimachine_insurance',
    'moped_policy_contribution': 'moped_insurance',
    'life_insurance_contribution': 'life_insurance',
    'private_accident_insurance_contribution': 'private_accident_insurance',
    'family_accident_insurance_contribution': 'family_accident_insurance',
    'disability_insurance_contribution': 'disability_insurance',
    'fire_insurance_contribution': 'fire_insurance',
    'surfboard_insurance_contribution': 'surfboard_insurance',
    'boat_insurance_contribution': 'boat_insurance',
    'bicycle_insurance_contribution': 'bicycle_insurance',
    'property_insurance_contribution': 'property_insurance',
    'social_security_insurance_contribution': 'social_security_insurance',
    # Insurance
    'number_private_third_party_insurance': 'private_third_party_policies_count',
    'number_business_third_party_insurance': 'business_third_party_policies_count',
    'number_agricultural_third_party_insurance': 'agricultural_third_party_policies_count',
    'number_car_policies': 'car_policy_count',
    'number_delivery_van_policies': 'van_policy_count',
    'number_motorcycle_scooter_policies': 'bike_policy_count',
    'number_lorry_policies': 'lorry_policy_count',
    'number_trailer_policies': 'trailer_policy_count',
    'number_tractor_policies': 'tractor_policy_count',
    'number_agricultural_machine_policies': 'agrimachine_policy_count',
    'number_moped_policies': 'moped_policy_count',
    'number_life_insurances': 'life_insurance_count',
    'number_private_accident_insurances': 'personal_accident_count',
    'number_family_accident_insurances': 'family_accident_count',
    'number_disability_insurances': 'disability_insurance_count',
    'number_fire_insurances': 'fire_insurance_count',
    'number_surfboard_insurances': 'surfboard_insurance_count',
    'number_boat_insurances': 'boat_insurance_count',
    'number_bicycle_insurances': 'bicycle_insurance_count',
    'number_property_insurances': 'property_insurance_count',
    'number_social_security_insurances': 'social_security_count',
    'number_mobile_home_policies': 'mobile_home_policy_count',
    'mobile_home_policies': 'mobile_home_policy'
}

# Store original column names before cleaning
original_column_names = insurance.columns.tolist()

# Clean column names and rename them according to the mapping
insurance.columns = insurance.columns.str.lower().str.replace(' ', '_')

# Store mapping between original and cleaned names
for i, orig_col in enumerate(original_column_names):
    cleaned_col = insurance.columns[i]
    original_names[cleaned_col] = orig_col

# Apply the column mapping
insurance = insurance.rename(columns=column_mapping)

# Update the original names dictionary to reflect the final mapping
for old_col, new_col in column_mapping.items():
    if old_col in original_names:
        original_names[new_col] = original_names[old_col]
    else:
        original_names[new_col] = old_col

# Create a cleaned version of the dataframe with updated names
ins = insurance.copy()

# Function to check if a float is an integer
def is_integer_value(val):
    if isinstance(val, float):
        return val.is_integer()
    return False

# Function to format values properly
def format_value(val):
    if pd.isna(val):
        return "NA"
    elif is_integer_value(val):
        return int(val)
    elif isinstance(val, float):
        return round(val, 3)
    return val

# Check for missing data
missing_data = ins.isnull().sum()
missing_percentage = (missing_data / len(ins)) * 100

# Create a DataFrame to display missing data information
missing_data_ins = pd.DataFrame({'Missing Values': missing_data, 'Percentage': missing_percentage})
missing_data_ins['Percentage'] = missing_data_ins['Percentage'].apply(lambda x: f"{x:.1f}")
missing_data_ins = missing_data_ins[missing_data_ins['Missing Values'] > 0].sort_values(by='Percentage', ascending=False)

# insurance_data_dictionary.csv was transformed and formatted into the following table [see disclaimer]
ins.to_csv("/content/drive/MyDrive/Colab Notebooks/data_mining_2/insurance_2.csv", index=False)

## 1.2 Generate descriptions based on column names and values
#### Grouping Insurance-Related Variables for Predictive Analysis

The structured organization of insurance-related variables enhances customer profiling and behavior prediction within the Insurance_Data.csv dataset. Categorizing features—such as household demographics, education levels, occupation types, social class, housing ownership, vehicle ownership, income variables, financial indicators, and insurance policies—streamlines exploratory analysis and machine learning applications, improving interpretability and predictive accuracy. This segmentation supports enhanced customer profiling by identifying socioeconomic traits, aiding classification into predefined subtypes. It also improves feature selection, ensuring that supervised learning models operate with refined and relevant attributes for better predictive performance.

**Household Demographics**: Captures familial structures and living arrangements, which influence insurance needs:

- Customer classification and segmentation  
- Total properties owned by household  
- Average household size and primary age bracket  
- Lifestyle classification  
- Breakdown of household compositions (married, cohabiting, extended family, singles, households with and without children)  

**Education Levels**: Impacts financial literacy and investment behaviors, shaping insurance enrollment trends:

- Proportion of highly educated, moderately educated, and low education individuals  

**Occupation Categories**: Determines income stability and risk exposure, necessitating tailored insurance options:

- High-status professionals  
- Entrepreneurs  
- Farmers  
- Middle management  
- Skilled workers  
- Unskilled workers  

**Social Class Categories**: Reflects purchasing power and economic stratification, impacting insurance preferences:

- Classification into social classes (A, B1, B2, C, D)  

**Housing & Vehicle Ownership**: Dictates insurance coverage needs based on property assets and transportation:

- Housing status (rented vs. owned)  
- Vehicle ownership (single car, dual car, or none)  

**Health Insurance**: Analyzes insurance enrollment trends based on accessibility and private coverage:  

- National health insurance participation  
- Private health insurance enrollment  

**Income Variables**: Provides insights into purchasing power and policy affordability:  

- Income distribution across brackets (below 30K, 30K-45K, 45K-75K, 75K-122K, above 123K)  
- Average segment income  
- Purchasing power classification  

**Financial Contributions to Insurance**: Consolidates spending patterns and premium contributions:  

- Private, business, and agricultural third-party insurance payments  
- Vehicle-specific insurance contributions (car, van, bike, lorry, trailer, tractor, agricultural machines, mopeds)  
- Life, accident, disability, and fire insurance contributions  
- Recreational property and asset insurance (surfboard, boat, bicycle, home, social security policies)  

**Insurance Policies & Holdings**: Tracks consumer choices and coverage trends, offering direct analysis of market demand:  

- Total count of held insurance policies for third-party, vehicle, property, life, accident, disability, fire, and recreational insurance  
- Mobile home policy tracking    

This logical segmentation aligns with the project’s aim of analyzing insurance customer behavior by supporting enhanced profiling, improving feature selection, refining clustering strategies, and optimizing actionable insights for policy customization.

In [4]:
# Generate descriptions based on column names with enhanced details
def generate_description(col_name):
    descriptions = {
        # Household and customer demographics
        "customer_type": "Classification of the customer or customer segmentation category",
        "total_properties_owned": "Total number of properties owned by the household",
        "avg_household_size": "Number of people living in the household",
        "avg_age": "Age bracket of the household's primary members",
        "household_classification": "Lifestyle classification of the household",
        "married": "Number/proportion of married people in the segment (out of 10)",
        "living_together": "Number/proportion of unmarried cohabiting people (out of 10)",
        "extended_family_members": "Number/proportion of people in other family relationships (out of 10)",
        "singles": "Number/proportion of single people (out of 10)",
        "childless_households": "Number/proportion of households without children (out of 10)",
        "households_with_children": "Number/proportion of households with children (out of 10)",

        # Education Levels
        "highly_educated": "Number/proportion with high education (out of 10)",
        "moderately_educated": "Number/proportion with medium education (out of 10)",
        "low_education": "Number/proportion with low education (out of 10)",

        # Occupation Categories
        "high_status_professionals": "Number/proportion in high-status professions (out of 10)",
        "entrepreneurs": "Number/proportion who are entrepreneurs (out of 10)",
        "farmer": "Number/proportion who are farmers (out of 10)",
        "middle_management": "Number/proportion in middle management roles (out of 10)",
        "skilled_workers": "Number/proportion of skilled workers (out of 10)",
        "unskilled_workers": "Number/proportion of unskilled workers (out of 10)",

        # Social Class Categories
        "social_class_a": "Number/proportion in social class A (highest) (out of 10)",
        "social_class_b1": "Number/proportion in social class B1 (out of 10)",
        "social_class_b2": "Number/proportion in social class B2 (out of 10)",
        "social_class_c": "Number/proportion in social class C (out of 10)",
        "social_class_d": "Number/proportion in social class D (lowest) (out of 10)",

        # Housing and vehicle ownership
        "rented_house": "Number/proportion living in rented housing (out of 10)",
        "home_owner": "Number/proportion who own their home (out of 10)",
        "single_car": "Number/proportion owning one car (out of 10)",
        "dual_car": "Number/proportion owning two cars (out of 10)",
        "no_car": "Number/proportion owning no car (out of 10)",

        # Health insurance
        "national_health_insurance": "Number/proportion with national health insurance (out of 10)",
        "private_health_insurance": "Number/proportion with private health insurance (out of 10)",

        # Income Variables
        "income_below_30k": "Number/proportion with income below 30K (out of 10)",
        "income_30k_to_45k": "Number/proportion with income between 30K-45K (out of 10)",
        "income_45k_to_75k": "Number/proportion with income between 45K-75K (out of 10)",
        "income_75k_to_122k": "Number/proportion with income between 75K-122K (out of 10)",
        "income_above_123k": "Number/proportion with income above 123K (out of 10)",
        "mean_income": "Average income value of the segment",
        "purchasing_power_group": "Purchasing power classification category",

        # Financial Contributions to Insurance
        "private_third_party": "Amount contributed to private third-party insurance",
        "business_third_party": "Amount contributed to business third-party insurance",
        "agricultural_third_party": "Amount contributed to agricultural third-party insurance",
        "car_insurance": "Amount contributed to car insurance policies",
        "van_insurance": "Amount contributed to delivery van insurance policies",
        "bike_insurance": "Amount contributed to motorcycle/scooter insurance policies",
        "lorry_insurance": "Amount contributed to lorry insurance policies",
        "trailer_insurance": "Amount contributed to trailer insurance policies",
        "tractor_insurance": "Amount contributed to tractor insurance policies",
        "agrimachine_insurance": "Amount contributed to agricultural machine insurance policies",
        "moped_insurance": "Amount contributed to moped insurance policies",
        "life_insurance": "Amount contributed to life insurance policies",
        "private_accident_insurance": "Amount contributed to personal accident insurance policies",
        "family_accident_insurance": "Amount contributed to family accident insurance policies",
        "disability_insurance": "Amount contributed to disability insurance policies",
        "fire_insurance": "Amount contributed to fire insurance policies",
        "surfboard_insurance": "Amount contributed to surfboard insurance policies",
        "boat_insurance": "Amount contributed to boat insurance policies",
        "bicycle_insurance": "Amount contributed to bicycle insurance policies",
        "property_insurance": "Amount contributed to property insurance policies",
        "social_security_insurance": "Amount contributed to social security insurance policies",

        # Insurance Policy Counts
        "private_third_party_policies_count": "Total count of private third-party insurance policies held",
        "business_third_party_policies_count": "Total count of business third-party insurance policies held",
        "agricultural_third_party_policies_count": "Total count of agricultural third-party insurance policies held",
        "car_policy_count": "Total count of car insurance policies held",
        "van_policy_count": "Total count of delivery van insurance policies held",
        "bike_policy_count": "Total count of motorcycle/scooter insurance policies held",
        "lorry_policy_count": "Total count of lorry insurance policies held",
        "trailer_policy_count": "Total count of trailer insurance policies held",
        "tractor_policy_count": "Total count of tractor insurance policies held",
        "agrimachine_policy_count": "Total count of agricultural machine insurance policies held",
        "moped_policy_count": "Total count of moped insurance policies held",
        "life_insurance_count": "Total count of life insurance policies held",
        "personal_accident_count": "Total count of personal accident insurance policies held",
        "family_accident_count": "Total count of family accident insurance policies held",
        "disability_insurance_count": "Total count of disability insurance policies held",
        "fire_insurance_count": "Total count of fire insurance policies held",
        "surfboard_insurance_count": "Total count of surfboard insurance policies held",
        "boat_insurance_count": "Total count of boat insurance policies held",
        "bicycle_insurance_count": "Total count of bicycle insurance policies held",
        "property_insurance_count": "Total count of property insurance policies held",
        "social_security_count": "Total count of social security insurance policies held",
        "mobile_home_policy_count": "Total count of mobile home insurance policies held",
        "mobile_home_policy": "Do or Don't have a mobile home insurance policy"
    }
    return descriptions.get(col_name, f"Represents {col_name.replace('_', ' ')}")

## 1.3 Generate descriptive statistics

In [5]:
# Generate descriptive statistics
descriptive_stats = insurance.describe(include='all').transpose()

## 1.4 Extracting example values from columns and Data Dictionary Compilation

In [6]:
# Create data dictionary
data_dictionary = []
for col in ins.columns:
    data_type = str(ins.dtypes[col])
    description = generate_description(col)
    # Example values (up to 5 unique examples formatted appropriately)
    example_values = []
    # Access unique values only if it's a Series
    if isinstance(ins[col], pd.Series):
        unique_vals = ins[col].dropna().unique()[:5]
        for val in unique_vals:
            example_values.append(str(format_value(val)))
    else:
        example_values = ["Error getting examples (not a Series)"] # [See Disclaimer]

    # Calculate statistics for each column
    count = len(ins)

    # Handle different data types appropriately
    if data_type in ['object', 'category']:
        mean = ins[col].mode()[0] if not ins[col].mode().empty else "Na"
        min_val = ins[col].dropna().min() if not ins[col].dropna().empty else "Na"
        max_val = ins[col].dropna().max() if not ins[col].dropna().empty else "Na"
        std_dev = "Na"
        # For categorical data, percentiles don't make sense
        percentile_25 = "Na"
        percentile_50 = "Na"
        percentile_75 = "Na"
    else:
        # For numeric columns, calculate statistics
        stats = ins[col].describe()
        mean = format_value(stats.get('mean', "Na"))
        min_val = format_value(stats.get('min', "Na"))
        max_val = format_value(stats.get('max', "Na"))
        std_dev = format_value(stats.get('std', "Na"))
        percentile_25 = format_value(stats.get('25%', "Na"))
        percentile_50 = format_value(stats.get('50%', "Na"))
        percentile_75 = format_value(stats.get('75%', "Na"))

    missing_values = ins[col].isna().sum()
    missing_percent = round((missing_values / len(ins)) * 100, 2) if len(ins) > 0 else 0
    data_dictionary.append({
        "Variable": col,
        "Original Variable": original_names.get(col, "Unknown"),
        "Data Type": data_type,
        "Number of Entries": count,
        "Missing Values": f"{missing_values} ({missing_percent}%)",
        "Description": description,
        "Example Values": ", ".join(example_values),
        "Mean": mean,
        "Min": min_val,
        "Max": max_val,
        "Std Dev": std_dev,
        "25%": percentile_25,
        "50%": percentile_50,
        "75%": percentile_75
    })
# Convert to a DataFrame for tabular representation
data_dictionary_df = pd.DataFrame(data_dictionary)
# Save the DataFrame as a CSV file
data_dictionary_df.to_csv("insurance_data_dictionary.csv", index=False)

----
# Insurance Data Dictionary

| Variable | Original Variable | Data Type | Number of Entries | Missing Values | Description | Example Values | Mean | Min | Max | Std Dev | 25% | 50% | 75% |
|----------|------------------|-----------|------------------|---------------|-------------|---------------|------|-----|-----|---------|-----|-----|-----|
| **customer_type** | customer_type | object | 5521 | 0 (0.0%) | Classification of the customer or customer segmentation category | Rural & Low-income, Middle-Class Families, Young & Low-income, Seniors & Retired, Wealthy & Affluent | Rural & Low-income | Middle-Class Families | Young & Low-income | Na | Na | Na | Na |
| **total_properties_owned** | number_of_houses | int64 | 5521 | 0 (0.0%) | Total number of properties owned by the household | 1, 2, 3, 10, 5 | 1.111 | 1 | 10 | 0.41 | 1 | 1 | 1 |
| **avg_household_size** | avg_household_size | int64 | 5521 | 0 (0.0%) | Number of people living in the household | 3, 2, 4, 1, 5 | 2.681 | 1 | 5 | 0.79 | 2 | 3 | 3 |
| **avg_age** | avg_age | object | 5521 | 14 (0.25%) | Age bracket of the household's primary members | 30-40 years, 40-50 years, 20-30 years, 50-60 years, 60-70 years | 40-50 years | 20-30 years | 70-80 years | Na | Na | Na | Na |
| **household_classification** | household_profile | object | 5521 | 0 (0.0%) | Lifestyle classification of the household | Family with Grown-Ups, Average Family, Farmers, Living Well, Conservative Families | Family with Grown-Ups | Average Family | Successful Hedonists | Na | Na | Na | Na |
| **married** | married | int64 | 5521 | 0 (0.0%) | Number/proportion of married people in the segment (out of 10) | 7, 6, 3, 5, 0 | 6.188 | 0 | 9 | 1.903 | 5 | 6 | 7 |
| **living_together** | living_together | int64 | 5521 | 0 (0.0%) | Represents living togethers | 0, 2, 1, 6, 4 | 0.883 | 0 | 7 | 0.967 | 0 | 1 | 1 |
| **extended_family_members** | other_relation | int64 | 5521 | 0 (0.0%) | Number/proportion of people in other family relationships (out of 10) | 2, 4, 3, 0, 1 | 2.286 | 0 | 9 | 1.714 | 1 | 2 | 3 |
| **singles** | singles | int64 | 5521 | 0 (0.0%) | Number/proportion of single people (out of 10) | 1, 0, 4, 2, 3 | 1.88 | 0 | 9 | 1.795 | 0 | 2 | 3 |
| **childless_households** | household_without_children | int64 | 5521 | 0 (0.0%) | Number/proportion of households without children (out of 10) | 2, 4, 3, 5, 6 | 3.235 | 0 | 9 | 1.62 | 2 | 3 | 4 |
| **households_with_children** | household_with_children | int64 | 5521 | 0 (0.0%) | Number/proportion of households with children (out of 10) | 6, 5, 2, 4, 3 | 4.303 | 0 | 9 | 2.007 | 3 | 4 | 6 |
| **highly_educated** | high_education_level | int64 | 5521 | 0 (0.0%) | Number/proportion with high education (out of 10) | 1, 0, 3, 5, 4 | 1.46 | 0 | 9 | 1.615 | 0 | 1 | 2 |
| **moderately_educated** | medium_education_level | int64 | 5521 | 0 (0.0%) | Number/proportion with medium education (out of 10) | 2, 5, 4, 3, 1 | 3.356 | 0 | 9 | 1.764 | 2 | 3 | 4 |
| **low_education** | low_education_level | int64 | 5521 | 0 (0.0%) | Number/proportion with low education (out of 10) | 7, 4, 2, 0, 5 | 4.57 | 0 | 9 | 2.298 | 3 | 5 | 6 |
| **high_status_professionals** | high_status | int64 | 5521 | 0 (0.0%) | Number/proportion in high-status professions (out of 10) | 1, 0, 4, 2, 3 | 1.891 | 0 | 9 | 1.794 | 0 | 2 | 3 |
| **entrepreneurs** | entrepreneur | int64 | 5521 | 0 (0.0%) | Number/proportion who are entrepreneurs (out of 10) | 0, 5, 1, 2, 3 | 0.397 | 0 | 5 | 0.774 | 0 | 0 | 1 |
| **farmer** | farmer | int64 | 5521 | 0 (0.0%) | Number/proportion who are farmers (out of 10) | 1, 0, 4, 3, 2 | 0.522 | 0 | 9 | 1.057 | 0 | 0 | 1 |
| **middle_management** | middle_management | int64 | 5521 | 0 (0.0%) | Number/proportion in middle management roles (out of 10) | 2, 5, 7, 3, 0 | 2.907 | 0 | 9 | 1.843 | 2 | 3 | 4 |
| **skilled_workers** | skilled_labourers | int64 | 5521 | 0 (0.0%) | Number/proportion of skilled workers (out of 10) | 5, 0, 1, 2, 8 | 2.22 | 0 | 9 | 1.729 | 1 | 2 | 3 |
| **unskilled_workers** | unskilled_labourers | int64 | 5521 | 0 (0.0%) | Number/proportion of unskilled workers (out of 10) | 2, 4, 0, 5, 1 | 2.306 | 0 | 9 | 1.694 | 1 | 2 | 3 |
| **social_class_a** | social_class_a | int64 | 5521 | 0 (0.0%) | Number/proportion in social class A (highest) (out of 10) | 1, 0, 3, 9, 2 | 1.619 | 0 | 9 | 1.721 | 0 | 1 | 2 |
| **social_class_b1** | social_class_b1 | int64 | 5521 | 0 (0.0%) | Number/proportion in social class B1 (out of 10) | 1, 2, 5, 0, 3 | 1.611 | 0 | 9 | 1.329 | 1 | 2 | 2 |
| **social_class_b2** | social_class_b2 | int64 | 5521 | 0 (0.0%) | Number/proportion in social class B2 (out of 10) | 2, 3, 0, 1, 4 | 2.205 | 0 | 9 | 1.534 | 1 | 2 | 3 |
| **social_class_c** | social_class_c | int64 | 5521 | 0 (0.0%) | Number/proportion in social class C (out of 10) | 6, 5, 4, 0, 8 | 3.756 | 0 | 9 | 1.935 | 2 | 4 | 5 |
| **social_class_d** | social_class_d | int64 | 5521 | 0 (0.0%) | Number/proportion in social class D (lowest) (out of 10) | 1, 0, 2, 5, 3 | 1.066 | 0 | 9 | 1.3 | 0 | 1 | 2 |
| **rented_house** | rented_house | int64 | 5521 | 0 (0.0%) | Number/proportion living in rented housing (out of 10) | 1, 2, 7, 5, 4 | 4.244 | 0 | 9 | 3.087 | 2 | 4 | 7 |
| **home_owner** | home_owner | int64 | 5521 | 0 (0.0%) | Number/proportion who own their home (out of 10) | 8, 7, 2, 4, 5 | 4.764 | 0 | 9 | 3.088 | 2 | 5 | 7 |
| **single_car** | owns_one_car | int64 | 5521 | 0 (0.0%) | Number/proportion owning one car (out of 10) | 8, 7, 9, 6, 5 | 6.037 | 0 | 9 | 1.551 | 5 | 6 | 7 |
| **dual_car** | owns_two_cars | int64 | 5521 | 0 (0.0%) | Number/proportion owning two cars (out of 10) | 0, 1, 2, 3, 4 | 1.32 | 0 | 7 | 1.205 | 0 | 1 | 2 |
| **no_car** | owns_no_car | int64 | 5521 | 0 (0.0%) | Number/proportion owning no car (out of 10) | 1, 2, 0, 3, 4 | 1.96 | 0 | 9 | 1.596 | 1 | 2 | 3 |
| **national_health_insurance** | national_health_insurance | int64 | 5521 | 0 (0.0%) | Number/proportion with national health insurance (out of 10) | 8, 6, 9, 7, 5 | 6.276 | 0 | 9 | 1.976 | 5 | 7 | 8 |
| **private_health_insurance** | private_health_insurance | int64 | 5521 | 0 (0.0%) | Number/proportion with private health insurance (out of 10) | 1, 3, 0, 2, 4 | 2.73 | 0 | 9 | 1.979 | 1 | 2 | 4 |
| **income_below_30k** | income_less_than_30k | int64 | 5521 | 0 (0.0%) | Number/proportion with income below 30K (out of 10) | 0, 2, 4, 1, 5 | 2.571 | 0 | 9 | 2.085 | 1 | 2 | 4 |
| **income_30k_to_45k** | income_30k_to_45k | int64 | 5521 | 0 (0.0%) | Number/proportion with income between 30K-45K (out of 10) | 4, 0, 5, 2, 3 | 3.531 | 0 | 9 | 1.879 | 2 | 4 | 5 |
| **income_45k_to_75k** | income_45k_to_75k | int64 | 5521 | 0 (0.0%) | Number/proportion with income between 45K-75K (out of 10) | 5, 0, 3, 9, 1 | 2.738 | 0 | 9 | 1.932 | 1 | 3 | 4 |
| **income_75k_to_122k** | income_75k_to_122k | int64 | 5521 | 0 (0.0%) | Number/proportion with income between 75K-122K (out of 10) | 0, 2, 1, 4, 3 | 0.797 | 0 | 9 | 1.166 | 0 | 0 | 1 |
| **income_above_123k** | income_above_123k | int64 | 5521 | 0 (0.0%) | Number/proportion with income above 123K (out of 10) | 0, 2, 1, 3, 4 | 0.203 | 0 | 7 | 0.545 | 0 | 0 | 0 |
| **mean_income** | average_income | int64 | 5521 | 0 (0.0%) | Average income value of the segment | 4, 5, 3, 6, 2 | 3.788 | 0 | 9 | 1.317 | 3 | 4 | 4 |
| **purchasing_power_group** | purchasing_power_class | int64 | 5521 | 0 (0.0%) | Purchasing power classification category | 3, 4, 5, 7, 2 | 4.244 | 1 | 8 | 2.003 | 3 | 4 | 6 |
| **private_third_party** | private_third_party_insurance_contribution | object | 5521 | 9 (0.16%) | Amount contributed to private third-party insurance | 0, 50-99, Jan-49, 100-199 | 0 | 0 | Jan-49 | Na | Na | Na | Na |
| **business_third_party** | business_third_party_insurance_contribution | object | 5521 | 0 (0.0%) | Amount contributed to business third-party insurance | 0, Jan-49, 100-199, 200-499, 50-99 | 0 | 0 | Jan-49 | Na | Na | Na | Na |
| **agricultural_third_party** | agricultural_third_party_insurance_contribution | object | 5521 | 0 (0.0%) | Amount contributed to agricultural third-party insurance | 0, 100-199, 200-499, 50-99 | 0 | 0 | 50-99 | Na | Na | Na | Na |
| **car_insurance** | car_policy_contribution | int64 | 5521 | 0 (0.0%) | Amount contributed to car insurance policies | 6, 0, 5, 7, 8 | 2.98 | 0 | 8 | 2.921 | 0 | 5 | 6 |
| **van_insurance** | delivery_van_policy_contribution | float64 | 5521 | 19 (0.34%) | Amount contributed to delivery van insurance policies | 0, 5, 6, 7 | 0.051 | 0 | 7 | 0.546 | 0 | 0 | 0 |
| **bike_insurance** | motorcycle_scooter_policy_contribution | int64 | 5521 | 0 (0.0%) | Amount contributed to motorcycle/scooter insurance policies | 0, 4, 5, 6, 7 | 0.174 | 0 | 7 | 0.892 | 0 | 0 | 0 |
| **lorry_insurance** | lorry_policy_contribution | int64 | 5521 | 0 (0.0%) | Amount contributed to lorry insurance policies | 0, 6, 4, 9 | 0.01 | 0 | 9 | 0.251 | 0 | 0 | 0 |
| **trailer_insurance** | trailer_policy_contribution | float64 | 5521 | 19 (0.34%) | Amount contributed to trailer insurance policies | 0, 2, 1, 3, 5 | 0.021 | 0 | 5 | 0.212 | 0 | 0 | 0 |
| **tractor_insurance** | tractor_policy_contribution | int64 | 5521 | 0 (0.0%) | Amount contributed to tractor insurance policies | 0, 3, 5, 4, 6 | 0.095 | 0 | 6 | 0.614 | 0 | 0 | 0 |
| **agrimachine_insurance** | agricultural_machine_policy_contribution | int64 | 5521 | 0 (0.0%) | Amount contributed to agricultural machine insurance policies | 0, 2, 6, 4, 3 | 0.014 | 0 | 6 | 0.235 | 0 | 0 | 0 |
| **moped_insurance** | moped_policy_contribution | float64 | 5521 | 52 (0.94%) | Amount contributed to moped insurance policies | 0, 3.251, 3.34, 3.915, 3.834 | 0.249 | 0 | 6.059 | 0.939 | 0 | 0 | 0 |
| **life_insurance** | life_insurance_contribution | float64 | 5521 | 60 (1.09%) | Amount contributed to life insurance policies | 0, 4.019, 4.91, 4.756, 3.698 | 0.225 | 0 | 9.048 | 1.018 | 0 | 0 | 0 |
| **private_accident_insurance** | private_accident_insurance_contribution | float64 | 5521 | 52 (0.94%) | Amount contributed to personal accident insurance policies | 0, 2.752, 2.651, 2.565, 2.236 | 0.016 | 0 | 6.787 | 0.239 | 0 | 0 | 0 |
| **family_accident_insurance** | family_accident_insurance_contribution | float64 | 5521 | 52 (0.94%) | Amount contributed to family accident insurance policies | 0, 2.107, 2.284, 3.801, 3.547 | 0.017 | 0 | 3.987 | 0.229 | 0 | 0 | 0 |
| **disability_insurance** | disability_insurance_contribution | float64 | 5521 | 72 (1.3%) | Amount contributed to disability insurance policies | 0, 6.198, 6.293, 6.414, 4.331 | 0.027 | 0 | 7.974 | 0.421 | 0 | 0 | 0 |
| **fire_insurance** | fire_insurance_contribution | float64 | 5521 | 55 (1.0%) | Amount contributed to fire insurance policies | 5.681, 2.022, 2.482, 2.427, 6.11 | 2.14 | 0 | 62.697 | 2.553 | 0 | 2.179 | 4.067 |
| **surfboard_insurance** | surfboard_insurance_contribution | float64 | 5521 | 52 (0.94%) | Amount contributed to surfboard insurance policies | 0, 1.766, 1.467, 3.125 | 0.001 | 0 | 3.125 | 0.052 | 0 | 0 | 0 |
| **boat_insurance** | boat_insurance_contribution | float64 | 5521 | 52 (0.94%) | Amount contributed to boat insurance policies | 0, 1.686, 4.538, 1.842, 6.382 | 0.021 | 0 | 6.582 | 0.301 | 0 | 0 | 0 |
| **bicycle_insurance** | bicycle_insurance_contribution | float64 | 5521 | 52 (0.94%) | Amount contributed to bicycle insurance policies | 0, 1.549, 1.239, 1.851, 1.348 | 0.04 | 0 | 1.968 | 0.246 | 0 | 0 | 0 |
| **property_insurance** | property_insurance_contribution | float64 | 5521 | 52 (0.94%) | Amount contributed to property insurance policies | 0, 2.745, 2.13, 1.089, 3.267 | 0.02 | 0 | 6.754 | 0.25 | 0 | 0 | 0 |
| **social_security_insurance** | social_security_insurance_contribution | float64 | 5521 | 52 (0.94%) | Amount contributed to social security insurance policies | 0, 4.804, 4.69, 4.381, 4.199 | 0.055 | 0 | 5.707 | 0.471 | 0 | 0 | 0 |
| **private_third_party_policies_count** | number_private_third_party_insurance | float64 | 5521 | 52 (0.94%) | Total count of private third-party insurance policies held | 0, 2, 1 | 0.404 | 0 | 2 | 0.493 | 0 | 0 | 1 |
| **business_third_party_policies_count** | number_business_third_party_insurance | float64 | 5521 | 52 (0.94%) | Total count of business third-party insurance policies held | 0, 1, 5 | 0.015 | 0 | 5 | 0.136 | 0 | 0 | 0 |
| **agricultural_third_party_policies_count** | number_agricultural_third_party_insurance | float64 | 5521 | 52 (0.94%) | Total count of agricultural third-party insurance policies held | 0, 1 | 0.021 | 0
| **car_policy_count** | number_car_policies | int64 | 5521 | 0 (0.0%) | Total count of car insurance policies held | 1, 0, 2, 7, 3 | 0.564 | 0 | 7 | 0.605 | 0 | 1 | 1 |
| **van_policy_count** | number_delivery_van_policies | int64 | 5521 | 0 (0.0%) | Total count of delivery van insurance policies held | 0, 1, 4, 2, 3 | 0.011 | 0 | 4 | 0.133 | 0 | 0 | 0 |
| **bike_policy_count** | number_motorcycle_scooter_policies | int64 | 5521 | 0 (0.0%) | Total count of motorcycle/scooter insurance policies held | 0, 1, 2 | 0.039 | 0 | 2 | 0.203 | 0 | 0 | 0 |
| **lorry_policy_count** | number_lorry_policies | int64 | 5521 | 0 (0.0%) | Total count of lorry insurance policies held | 0, 1, 2, 3 | 0.002 | 0 | 3 | 0.065 | 0 | 0 | 0 |
| **trailer_policy_count** | number_trailer_policies | float64 | 5521 | 32 (0.58%) | Total count of trailer insurance policies held | 0, 1, 2, 3 | 0.012 | 0 | 3 | 0.126 | 0 | 0 | 0 |
| **tractor_policy_count** | number_tractor_policies | float64 | 5521 | 32 (0.58%) | Total count of tractor insurance policies held | 0, 1, 2, 3, 4 | 0.035 | 0 | 4 | 0.246 | 0 | 0 | 0 |
| **agrimachine_policy_count** | number_agricultural_machine_policies | float64 | 5521 | 32 (0.58%) | Total count of agricultural machine insurance policies held | 0, 1, 3, 2, 6 | 0.007 | 0 | 6 | 0.128 | 0 | 0 | 0 |
| **moped_policy_count** | number_moped_policies | float64 | 5521 | 32 (0.58%) | Total count of moped insurance policies held | 0, 1, 2 | 0.07 | 0 | 2 | 0.265 | 0 | 0 | 0 |
| **life_insurance_count** | number_life_insurances | float64 | 5521 | 32 (0.58%) | Total count of life insurance policies held | 0, 1, 2, 8, 3 | 0.076 | 0 | 8 | 0.375 | 0 | 0 | 0 |
| **personal_accident_count** | number_private_accident_insurances | float64 | 5521 | 32 (0.58%) | Total count of personal accident insurance policies held | 0, 1 | 0.005 | 0 | 1 | 0.073 | 0 | 0 | 0 |
| **family_accident_count** | number_family_accident_insurances | float64 | 5521 | 32 (0.58%) | Total count of family accident insurance policies held | 0, 1 | 0.006 | 0 | 1 | 0.077 | 0 | 0 | 0 |
| **disability_insurance_count** | number_disability_insurances | int64 | 5521 | 0 (0.0%) | Total count of disability insurance policies held | 0, 1, 2 | 0.005 | 0 | 2 | 0.079 | 0 | 0 | 0 |
| **fire_insurance_count** | number_fire_insurances | int64 | 5521 | 0 (0.0%) | Total count of fire insurance policies held | 1, 0, 2, 3, 4 | 0.569 | 0 | 7 | 0.56 | 0 | 1 | 1 |
| **surfboard_insurance_count** | number_surfboard_insurances | int64 | 5521 | 0 (0.0%) | Total count of surfboard insurance policies held | 0, 1 | 0.001 | 0 | 1 | 0.023 | 0 | 0 | 0 |
| **boat_insurance_count** | number_boat_insurances | int64 | 5521 | 0 (0.0%) | Total count of boat insurance policies held | 0, 2, 1 | 0.006 | 0 | 2 | 0.081 | 0 | 0 | 0 |
| **bicycle_insurance_count** | number_bicycle_insurances | int64 | 5521 | 0 (0.0%) | Total count of bicycle insurance policies held | 0, 1, 2, 3 | 0.033 | 0 | 3 | 0.215 | 0 | 0 | 0 |
| **property_insurance_count** | number_property_insurances | float64 | 5521 | 49 (0.89%) | Total count of property insurance policies held | 0, 1, 2 | 0.008 | 0 | 2 | 0.092 | 0 | 0 | 0 |
| **social_security_count** | number_social_security_insurances | float64 | 5521 | 49 (0.89%) | Total count of social security insurance policies held | 0, 1, 2 | 0.014 | 0 | 2 | 0.12 | 0 | 0 | 0 |
| **mobile_home_policy_count** | number_mobile_home_policies | float64 | 5521 | 74 (1.34%) | Total count of mobile home insurance policies held | 0, 1 | 0.06 | 0 | 1 | 0.238 | 0 | 0 | 0 |
| **mobile_home_policy** | mobile_home_policies | object | 5521 | 65 (1.18%) | Do or Don't have mobile home insurance policy | No Policy, Has Policy | No Policy | Has Policy | No Policy | Na | Na | Na | Na |